![Platform+](images/banner.png)

# Lab 7: Getting Data In
## Stage → Format → COPY INTO → Query → Monitor → Govern

> **Persona track: Platform Admin** — same lab, your angle.
>
> Getting data in is only step one. An admin owns the *plumbing* around the load — it has to be reliable, observable, and secure. In this lab you'll follow one dataset through the admin's loop: **provision the stage & format → load it → monitor load history (and alert on failures) → govern who can access it.**

:point_right: **In this lesson, you'll bring two datasets into Snowflake: loyalty sign-ups from an internal stage, and a supplier product catalog from a dedicated supplier stage. You'll define file formats, load both files with COPY INTO, then take the admin's view: diagnose a broken file format, audit and monitor load history, stand up an alert for failed loads, and manage access to the objects that make loading (and unloading) work.**

The scenario: The Roastery loyalty programme has been running for months — sign-up data lives in an internal stage. Meanwhile, your coffee bean supplier drops quarterly product catalog updates to a separate supplier stage. As the Platform Admin you're not just loading the data — you're responsible for making ingestion **reliable, observable, and securely accessible** to the right roles.

We'll cover:
1. Inspecting what's in a stage before loading
2. Defining a CSV file format and loading with COPY INTO
3. Connecting to the supplier stage and loading the product catalog
4. Catching bad data with VALIDATION_MODE
5. Your angle — diagnose a broken file format, audit & monitor loads (with an alert), and manage access to loading objects

:books: **By the end you'll be able to:**
- Use `LIST` and `SELECT $1, $2, $3` to inspect a staged file before loading
- Create a FILE FORMAT and use it with COPY INTO
- Run a dry-run load with VALIDATION_MODE to catch errors before they land
- Diagnose and fix the classic "comma inside a quoted field" load failure
- Query and build a monitoring view over `COPY_HISTORY` — and know what it does *not* record
- Create an alert that watches for failed/partial loads
- Grant the stage and file-format privileges a role needs to load and unload data

**Time:** ~50 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)
print('Your data-team role (for the access-control section): ' + my_analyst_role)


## Set your session context

Every query runs under a role (what you're allowed to do) and a warehouse (compute). The cells below create your personal warehouse and database if they don't exist yet, then switch into your own schema for this lab.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;
USE SCHEMA {{my_db}}.RAW;


---
## 1 · Inspect Before You Load

Good loading practice: always look at the raw file before writing a single line of SQL. As an admin, you're also the person who provisioned the stages — so understanding what's in them is both an operational check and a teaching moment for your data teams.

The stage `PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE` holds the loyalty sign-up data. Let's see what's there.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE;


> :mag_right: **Review the output.** You should see `loyalty_signups.csv`. Note the file size and `last_modified` timestamp — these are useful when debugging "why hasn't my data updated?"

**Reading from a stage without loading.** The `SELECT` below doesn't move any data into Snowflake — it streams the raw file from the stage and returns each row as text. Nothing is written anywhere.

The `$1`, `$2`, `$3` notation is **positional**: Snowflake maps each delimited field in the file to a numbered column — `$1` is the first field, `$2` the second, and so on. Because no **FILE FORMAT** has been applied yet, every field comes back as a raw string. Adding `$4` shows the *boundary*: an empty `$4` means field 4 doesn't exist, confirming exactly 3 columns. Use this to answer the three questions you need: *How many columns are there? Does the file have a header row? What's the delimiter?*

> 💡 **What's a FILE FORMAT?** A FILE FORMAT is the small set of instructions that tells Snowflake how to interpret the shape of a file: its delimiter, whether it has a header row, how nulls are written. You'll create one in the next step.

In [ ]:
-- Positional preview: $1/$2/$3 = the three fields, $4 empty = no 4th column (boundary check)
SELECT $1, $2, $3, $4
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
LIMIT 5;


> :mag_right: **Your turn — read the file.**  
> Before building the FILE FORMAT you need to answer three questions about this file:
> 1. **How many columns are there?** (Look at which positional columns have values — and which is empty.)
> 2. **Does the file have a header row?** (Look at row 1: is it data or labels?)
> 3. **What's the delimiter?** (The filename is a clue — what does `.csv` stand for?)
>
> <details><summary>Check your read</summary>
>
> **Three columns** — `$1`, `$2`, and `$3` all have values; `$4` is empty in every row, marking the column boundary.  
> **Row 1 is the header** — it shows `CUSTOMER_ID`, `SIGNUP_DATE`, `CHANNEL` rather than actual values. You'll need to skip it with `SKIP_HEADER=1`.  
> **Comma-delimited CSV** — `.csv` stands for comma-separated values, so the delimiter is a comma. `CSV` is exactly the value you'll enter for `TYPE` in the FILE FORMAT cell coming up next. The date format is `YYYY-MM-DD` and channel is a short string (`app`, `web`, `in-store`).
>
> That's all the information you need to write the FILE FORMAT.
> </details>

---
## 2 · Define the Format, Then COPY INTO

Snowflake uses **FILE FORMAT** objects to describe how a staged file is structured. You define it once and reuse it across COPY INTO statements.

**If you don't define a file format, Snowflake assumes CSV** — comma-delimited fields and newline-delimited rows. That default is fine for a simple comma-separated file, but the moment your file has a header row, a different delimiter, or a different type (like JSON), you have to spell it out.

**Why this matters for admins:** You'll often be the one defining FILE FORMAT objects for your team, or diagnosing why a team's format definition doesn't match their file. Understanding every option means you can resolve load issues before they become incidents.

> :warning: **The cell below is wrong on purpose.** It declares the format as `TYPE = JSON` with `SKIP_HEADER = 0`. Run it anyway — Snowflake will reject it. Read the error: it tells you an option isn't valid for JSON, which is your clue that this isn't a JSON file at all. It's a CSV with a header row. **Fix the two settings** to what you learned from the peek:
> - `TYPE` — the *kind* of file you're loading. The file extension and peek confirmed it's a comma-separated-values file, so this is `CSV` (not JSON).
> - `SKIP_HEADER` — how many rows at the top are labels rather than data. The peek showed a single header row, so this is `1` (not 0).
>
> The rest is already correct for you: `FIELD_DELIMITER=','` (fields split on commas), `TRIM_SPACE=TRUE` (strip stray spaces), `NULL_IF=('','NULL')` (treat blanks and the literal text `NULL` as null), and `EMPTY_FIELD_AS_NULL=TRUE`. Reference solution is just below the cell.

In [ ]:
-- WRONG ON PURPOSE: TYPE should not be JSON, and SKIP_HEADER should not be 0.
-- Run it, read the error, then fix TYPE and SKIP_HEADER (reference is just below the cell).
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.CSV_FMT
  TYPE           = JSON
  SKIP_HEADER    = 0
  FIELD_DELIMITER = ','
  TRIM_SPACE     = TRUE
  NULL_IF        = ('', 'NULL')
  EMPTY_FIELD_AS_NULL = TRUE;


> <details><summary>📖 Reference solution</summary>
>
>     TYPE        = CSV   -- comma-separated-values file (not JSON)
>     SKIP_HEADER = 1     -- skip the single header row (not 0)
>
> - **TYPE = CSV** — tells Snowflake to parse the file as delimited text.
> - **SKIP_HEADER = 1** — ignores row 1 (the column names) so it isn't loaded as a data row.
> - **FIELD_DELIMITER = ','** — fields are separated by commas.
> - **TRIM_SPACE / NULL_IF / EMPTY_FIELD_AS_NULL** — strip stray whitespace and turn blanks or the literal text `NULL` into real NULLs.
> </details>

### Now create the table to load into

The FILE FORMAT describes *how to read* the file. Next you need somewhere to put the data: a table whose columns line up with the file's fields. `COPY INTO` maps them **by position** — field 1 → column 1, field 2 → column 2, and so on — so the column order below must match the order you saw in the peek (`customer_id`, `signup_date`, `channel`), and each column needs a type that fits its values.

The types are already filled in for you, but you should understand why: `customer_id` is `INT` because the peek showed numeric IDs; `signup_date` is `DATE` so you can do date math on it later; `channel` is `VARCHAR(30)` because it holds short text values.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.LOYALTY_SIGNUPS (
  customer_id  INT,
  signup_date  DATE,
  channel      VARCHAR(30)
);


This is your first **COPY INTO** — the command that actually loads staged data into a table. It has a *lot* of optional parameters (error handling, file selection, transformations, validation). As the admin, you don't need to memorise them — but you should know they exist and how to look them up.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What are the main parameters of the Snowflake COPY INTO &lt;table&gt; command? Group them by purpose (file location, file format, error handling, load behaviour) and give a one-line description of each.</pre>

> :point_right: **Your turn.** The COPY INTO below is missing its file format name. Fill in the name of the format you just fixed (`FORMAT_NAME = ______`), then run it. Reference solution is just below the cell.

In [ ]:
COPY INTO {{my_db}}.RAW.LOYALTY_SIGNUPS
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
FILE_FORMAT = (FORMAT_NAME = ______);  -- fill in your file format name


> <details><summary>📖 Reference solution</summary>
>
>     FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT)
> </details>

In [ ]:
SELECT COUNT(*) AS rows_loaded FROM {{my_db}}.RAW.LOYALTY_SIGNUPS;


> :mag_right: **Review the output.** The COPY INTO result shows rows loaded, parse errors (should be 0), and file status (LOADED). Confirm the COUNT(*) matches.

> :point_right: **Your turn.** Run `SELECT * FROM {{my_db}}.RAW.LOYALTY_SIGNUPS LIMIT 10` to confirm column types loaded correctly.

In [ ]:
SELECT * FROM {{my_db}}.RAW.LOYALTY_SIGNUPS LIMIT 10;


---
## 3 · Connect to the Supplier Stage

The loyalty sign-ups came from an internal stage — data your team controls. But many real-world sources come from *outside*: a vendor, a partner, a cloud-based system writing files to its own bucket.

**The scenario:** Roastery's coffee bean suppliers send quarterly product catalog updates to a dedicated supplier stage. As the admin, you provisioned this stage. Now let's walk through loading from it.

Start by listing the stage to see what the supplier has dropped.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE;


> :mag_right: **Review the output.** You'll see the clean catalog `roastery_products_q4_2025.csv` (load this now), an errors file `roastery_products_q4_2025_errors.csv` (Section 4), a quoted-field file `roastery_products_q4_2025_quoted.csv` (your Part A challenge), and `roastery_loyalty_events.json` (Data Engineer notebook only — skip it here).

In [ ]:
SELECT $1, $2, $3, $4, $5, $6
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
LIMIT 4;


> :mag_right: **Review the output.** Row 1 is the header: `product_id`, `product_name`, `category`, `origin_country`, `cost_per_unit`, `active`. The data is clean — numeric IDs, decimal costs, TRUE/FALSE flags.

---

**You just listed an internal stage.** In production, a supplier wouldn't write to your internal stage — they'd write to their own S3 bucket and you'd point an **external stage** at it. Setting that up securely is squarely an admin job.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/integrations — we're using an internal stage here. Explain what we would need to change to point this at an external S3 bucket owned by a supplier. What is a storage integration, when is it required, and what trust policy step must be completed in AWS after running DESCRIBE INTEGRATION?</pre>

> 📌 **Note:** This lab uses an internal stage so you can run it without cloud infrastructure. When the **infrastructure administrator** provisions the supplier's S3 bucket, only the `CREATE STAGE` definition changes — the COPY INTO syntax you write below stays exactly the same.

**Same three steps as before.** Loading the supplier catalog follows the exact pattern you just used for the sign-ups: **(1)** define a file format that matches the file, **(2)** create the target table, **(3)** run COPY INTO. You've seen it once — now you'll build more of it yourself.

> 💡 **Why a second file format?** You already have `CSV_FMT` — but here you'll author a format from scratch for the practice. In real pipelines each source often gets its own named format, so this mirrors production.

> :point_right: **Your turn.** The file format below is only started for you — just the `CREATE` line and `TYPE = CSV`. This is a CSV with a header row, so add the option it needs to skip that header. (Everything else you might add is already the default, so `SKIP_HEADER` is the one line that matters here.) Reference solution is just below the cell.

In [ ]:
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.CSV_SUPPLIER_FMT
  TYPE = CSV
  -- your turn: add the option this file needs (it has a header row)


> <details><summary>📖 Reference solution</summary>
>
>     SKIP_HEADER = 1
> </details>

### Now create the target table

Same as before: the format describes *how to read* the file; the table is *where the rows land*. The columns below match the six fields you saw in the peek, in order, each with a type that fits its values (`INT` id, text names, `FLOAT` cost, `BOOLEAN` active flag). `COPY INTO` maps them by position, so the order matters.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS (
  product_id      INT,
  product_name    VARCHAR(100),
  category        VARCHAR(50),
  origin_country  VARCHAR(50),
  cost_per_unit   FLOAT,
  active          BOOLEAN
);


> :point_right: **Your turn — fill in the skeleton.** The three lines of a COPY INTO are laid out for you below; complete each blank: load `roastery_products_q4_2025.csv` from `@PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE` into `SUPPLIER_PRODUCTS`, using the `CSV_SUPPLIER_FMT` format you just built. Then run it. Reference solution is just below the cell.

In [ ]:
-- Your turn: complete each blank, then run.
COPY INTO ______                       -- the target table (SUPPLIER_PRODUCTS)
FROM ______                            -- the staged file (roastery_products_q4_2025.csv)
FILE_FORMAT = (FORMAT_NAME = ______);  -- the format you just built (CSV_SUPPLIER_FMT)


> <details><summary>📖 Reference solution</summary>
>
>     COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS
>     FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
>     FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_SUPPLIER_FMT);
> </details>

> :warning: **Getting "Numeric value 'product_id' is not recognized"?** That means the header row is being loaded as data — your `CSV_SUPPLIER_FMT` is missing `SKIP_HEADER = 1`. Fix the format above, then re-run the `COPY INTO` cell. Paste the error into CoCo if you want it to walk you through the fix.

> :mag_right: **Review the output.** The COPY INTO result above already confirmed the load — check its **`rows_parsed`** and **`rows_loaded`** (50 each, 0 errors). This cell spot-checks the actual rows so you can eyeball types and values before moving on.

In [ ]:
-- Spot-check the data itself (the COPY INTO output above already reported the row count)
SELECT * FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS LIMIT 5;


---
## 4 · Handle Load Errors

The supplier also sent a file with two rows where `cost_per_unit` contains text (`N/A`, `PENDING`) instead of a number. **VALIDATION_MODE** lets you do a dry-run — Snowflake checks every row but writes nothing. You see exactly which rows fail and why, before any data lands in your table.

**Why this matters for admins:** When a team reports "the product table is missing rows", VALIDATION_MODE is your first diagnostic tool. Run it against the source file; if errors appear, the problem is in the data — not the pipeline. If no errors appear, look at the FILE FORMAT definition.

In [ ]:
-- Dry-run: no data is loaded, but Snowflake reports every row that would fail
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_errors.csv
FILE_FORMAT  = (FORMAT_NAME = {{my_db}}.RAW.CSV_SUPPLIER_FMT)
VALIDATION_MODE = RETURN_ALL_ERRORS;


> :mag_right: **Review the output.** Two rows flagged. The `ERROR` column tells you which column failed and what value was received. The `ROW_NUMBER` column gives the exact line in the file. This is the information you'd send to the supplier to request a corrected file.

> :bulb: **A safe diagnostic.** A `VALIDATION_MODE` dry-run writes nothing to your table — and because nothing commits, it leaves **no trace in `COPY_HISTORY`** either. It's the ideal way to vet a file before you trust it; just remember there's no audit record that you ran it.

> 🛡️ **Your angle starts here.** Everything above was shared foundation — staging, formats, COPY INTO. From here on, you apply it as an admin: monitoring load health, controlling who can stage and load, and alerting when things break.

---
## 5 · Your angle — Platform Admin: diagnose, monitor, and control access

### Part A · Inspect stage & format objects — then fix a broken one

As an admin, you need operational visibility into what stages and file formats exist. These SHOW commands are your starting point.

In [ ]:
-- See all stages in the shared PLATPLUS_DB.ROASTERY_RAW schema
SHOW STAGES IN SCHEMA PLATPLUS_DB.ROASTERY_RAW;


> :mag_right: **Review the output.** Look at the `type` column — `INTERNAL` vs `EXTERNAL` (and `INTERNAL NO CSE` = an internal stage with server-side encryption only, no client-side encryption). The `url` column is empty for internal stages, populated for external ones. `has_credentials` tells you whether credentials are stored. The `owner` column shows who owns each stage — important for knowing who can grant access to it.

In [ ]:
-- See the file formats you've created in this lab
SHOW FILE FORMATS IN SCHEMA {{my_db}}.RAW;


In [ ]:
-- Inspect every option on a file format you defined
DESCRIBE FILE FORMAT {{my_db}}.RAW.CSV_SUPPLIER_FMT;


> :mag_right: **Review the DESCRIBE output.** Every FILE FORMAT property is shown with its current value and its default. This is the fastest way to verify a format is configured exactly as intended — especially when someone reports "the file format looks wrong" and you need to confirm or deny.

---
#### The file-format challenge 🔧

The supplier sent one more file this quarter: `roastery_products_q4_2025_quoted.csv`. A teammate tried to load it and it **failed**. As the admin, it's on your desk to diagnose.

Let's reproduce the failure, find the root cause, and fix it — the single most common CSV loading incident you'll ever handle.

First, a fresh landing table for the challenge. `SUPPLIER_PRODUCTS_QUOTED` has the same six columns as before — it's a separate target so the failed and then fixed loads don't disturb the clean `SUPPLIER_PRODUCTS` table you already loaded.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS_QUOTED (
  product_id      INT,
  product_name    VARCHAR(100),
  category        VARCHAR(50),
  origin_country  VARCHAR(50),
  cost_per_unit   FLOAT,
  active          BOOLEAN
);


**Run the load below with a standard CSV format.** It *will* fail — that's expected. Read the error carefully before moving on.

In [ ]:
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS_QUOTED
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_quoted.csv
FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1);


> :mag_right: **Read the error.** *"Number of columns in file (7) does not match that of the corresponding table (6)."* The file has the right 6 columns per row — so why is Snowflake seeing 7? Peek at the raw file to find out.

In [ ]:
-- Read the raw fields positionally. Watch what lands in $2, $3, and $7.
SELECT $1, $2, $3, $4, $5, $6, $7
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_quoted.csv
LIMIT 5;


**See the raw line.** Turn off delimiter parsing with `FIELD_DELIMITER='NONE'` so `$1` returns the whole line — now you can see the real delimiter and any quoting yourself.

In [ ]:
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.NO_DELIM_FMT TYPE='CSV' FIELD_DELIMITER='NONE';
SELECT $1 AS raw_line
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_quoted.csv
  (FILE_FORMAT => '{{my_db}}.RAW.NO_DELIM_FMT')
LIMIT 5;


> :mag_right: **There's the culprit.** A product name like `"Ethiopia Yirgacheffe, Grade 1"` contains a **comma inside the field**, wrapped in double quotes. Your format treats every comma as a delimiter, so that one value gets split across `$2` and `$3` — pushing the row to 7 fields.
>
> **The fix:** tell Snowflake that double-quoted text is a single field with `FIELD_OPTIONALLY_ENCLOSED_BY = '"'`.
>
> :point_right: **Your turn.** Add that option to the COPY below (replace `______`), then run it. Reference solution is just below the cell.
>
> :warning: **The tempting wrong fix.** The error message suggests `ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE`. **Don't.** That silences the error but still splits the row on the embedded comma — so the data loads *misaligned* (values shifted into the wrong columns) and truncated. You'd turn a loud, obvious failure into silent data corruption. Always fix the root cause.

In [ ]:
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS_QUOTED
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_quoted.csv
FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 ______);  -- add the option that handles quoted fields


> <details><summary>📖 Reference solution</summary>
>
>     FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
> </details>

In [ ]:
SELECT product_id, product_name, cost_per_unit
FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS_QUOTED
ORDER BY product_id;


> :mag_right: **Review the output.** 8 rows, and names like `Ethiopia Yirgacheffe, Grade 1` and `Chemex Pour-Over, 6-Cup` sit intact in a single `product_name` column — comma and all. Diagnosed and fixed.

> :bulb: **An admin insight for the next section.** That first attempt was a **hard abort** — the whole statement failed and loaded nothing. Hard aborts fail loudly at runtime but leave **no row in `COPY_HISTORY`**. The dangerous loads are the ones that *succeed quietly* while dropping or misaligning rows — those you can only catch by monitoring. That's Part B.

---
### Part B · Audit & monitor load history

`COPY_HISTORY` is your load audit trail — it records what was loaded, when, how many rows, and how many errors. But it only records **committed loads**. Two things you've already done this lab do **not** appear:
- the **VALIDATION_MODE** dry-run in Section 4 (wrote nothing), and
- the **hard-abort** first attempt in Part A (aborted before committing).

The loads you *most* need to catch are the ones that commit **with errors**. Let's create one, then audit it.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL (
  product_id      INT,
  product_name    VARCHAR(100),
  category        VARCHAR(50),
  origin_country  VARCHAR(50),
  cost_per_unit   FLOAT,
  active          BOOLEAN
);


> :warning: **A different file — and a "quiet" failure.** This loads `roastery_products_q4_2025_errors.csv`, where 2 rows have text (`N/A`, `PENDING`) in the numeric `cost_per_unit` column. The key clause is **`ON_ERROR = CONTINUE`**: it commits the 50 clean rows and silently skips the 2 bad ones. The load **succeeds** — which is exactly what makes it dangerous. Nobody gets an error; the rows just quietly go missing.

In [ ]:
-- ON_ERROR = CONTINUE commits the clean rows and skips the bad ones.
-- This is the "quiet" load: it succeeds, but with errors you must monitor for.
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_errors.csv
FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_SUPPLIER_FMT)
ON_ERROR = CONTINUE;


> :point_right: **So how would you even know?** The load didn't fail, so nothing alerts you. You ask the audit trail. `COPY_HISTORY` records every committed load with its parsed/loaded/error counts — the one place a partial load shows its face.

In [ ]:
-- The audit trail for your loads in the last 24 hours
SELECT
    TABLE_NAME,
    FILE_NAME,
    ROW_COUNT,
    ROW_PARSED,
    ERROR_COUNT,
    STATUS,
    LAST_LOAD_TIME
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => '{{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL',
    START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())))
ORDER BY LAST_LOAD_TIME DESC;


> :mag_right: **Review the output.** `ROW_COUNT` = 50 (committed), `ROW_PARSED` = 52 (read), `ERROR_COUNT` = 2, `STATUS` = `Partially loaded`. This row is exactly the kind of "quiet" problem load you need to notice.

> :bulb: **What `COPY_HISTORY` does NOT tell you: who ran the load.** There's no user/role column here. For "who loaded this?" you use `ACCESS_HISTORY` or `QUERY_HISTORY` (in the `SNOWFLAKE` shared database), which record the user and role per query.

#### Recover the failed rows 🔧

Monitoring told you *that* 2 rows failed. Now close the loop: get them in. Your instinct might be to just **re-run the load** — but that won't work, and the reason is worth knowing.

> :bulb: **Why you can't just re-run it.** Snowflake tracks which files a table has already loaded (load metadata, kept ~64 days). Re-run the exact same `COPY INTO` and you'll get **"Copy executed with 0 files processed"** — it refuses to reload the same file. Your options: `FORCE = TRUE` (reloads *everything* → duplicate rows — rarely what you want), load a *corrected* file under a new name, or **salvage the rejected rows you already have**. Let's do the last one — no new file needed.

In [ ]:
-- VALIDATE replays a past COPY and returns the rows it REJECTED.
-- '_last' = the most recent COPY in THIS session — run this right after the load,
-- before any other COPY runs (Part C loads again later).
SELECT ROW_NUMBER, COLUMN_NAME, ERROR, REJECTED_RECORD
FROM TABLE(VALIDATE({{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL, JOB_ID => '_last'));


> :mag_right: **Review the output.** Two rows — the `N/A` and `PENDING` cost rows. The star column is **`REJECTED_RECORD`**: it holds the *entire raw line* Snowflake couldn't load (e.g. `511,Chemex Pour-Over 6-Cup,Equipment,USA,N/A,TRUE`). Because you have the full record, you can repair and re-insert it without asking the supplier for anything.

> :point_right: **Your turn.** Parse each field out of `REJECTED_RECORD` and insert the two rows — applying a business rule to the column that broke: **an unknown cost becomes `NULL`** (load the row now, chase the real price later). One expression is blank — supply the salvaged cost. Reference solution is just below the cell.

In [ ]:
-- Salvage straight from REJECTED_RECORD: split the raw line back into columns,
-- fixing the value that failed. SPLIT_PART(line, ',', n) grabs the n-th field.
INSERT INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL
SELECT SPLIT_PART(rejected_record, ',', 1)::INT,      -- product_id
       SPLIT_PART(rejected_record, ',', 2),           -- product_name
       SPLIT_PART(rejected_record, ',', 3),           -- category
       SPLIT_PART(rejected_record, ',', 4),           -- origin_country
       ______,                                        -- the salvaged cost (unknown -> NULL)
       SPLIT_PART(rejected_record, ',', 6)::BOOLEAN   -- active
FROM TABLE(VALIDATE({{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL, JOB_ID => '_last'));


> <details><summary>📖 Reference solution</summary>
>
>     NULL
>
> Casting `'N/A'` or `'PENDING'` to a number would just fail again. Loading the row with a `NULL` cost gets the product into the table now; a downstream job (or the supplier's corrected file) can fill the real price later. This "load first, perfect later" move is central to resilient pipelines.
> </details>

In [ ]:
-- The loop is closed: 50 loaded + 2 recovered = 52
SELECT COUNT(*) AS total_rows FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL;


In [ ]:
SELECT product_id, product_name, category, cost_per_unit
FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL
WHERE product_id IN (511, 703);


> :mag_right: **Review the output.** `total_rows` = **52** — the 2 rows that quietly failed are now in, with `cost_per_unit = NULL` flagging them for follow-up. You've run the full admin recovery loop: **monitor** (`COPY_HISTORY`) → **diagnose** (`VALIDATE`) → **recover** (fix + insert) → **verify**.

> :bulb: **In production you'd automate the "load everything, quarantine the bad" version.** Instead of hand-repairing, teams often land raw data into an all-`VARCHAR` staging table (nothing ever rejects), then `TRY_CAST` in SQL — clean rows flow to the typed table, un-castable rows route to a `_QUARANTINE` table for review. Same idea, fully hands-off. Ask CoCo about the `TRY_CAST` quarantine pattern.

**Turn it into a monitoring view.** Rather than run that query per table by hand, wrap your key tables into a single view an admin can glance at. This one unions the load history of every table you've loaded this lab.

In [ ]:
CREATE OR REPLACE VIEW {{my_db}}.RAW.LOAD_MONITOR AS
SELECT TABLE_NAME, FILE_NAME, ROW_COUNT, ERROR_COUNT, STATUS, LAST_LOAD_TIME
  FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => '{{my_db}}.RAW.LOYALTY_SIGNUPS',          START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())))
UNION ALL
SELECT TABLE_NAME, FILE_NAME, ROW_COUNT, ERROR_COUNT, STATUS, LAST_LOAD_TIME
  FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => '{{my_db}}.RAW.SUPPLIER_PRODUCTS',         START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())))
UNION ALL
SELECT TABLE_NAME, FILE_NAME, ROW_COUNT, ERROR_COUNT, STATUS, LAST_LOAD_TIME
  FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => '{{my_db}}.RAW.SUPPLIER_PRODUCTS_QUOTED', START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())))
UNION ALL
SELECT TABLE_NAME, FILE_NAME, ROW_COUNT, ERROR_COUNT, STATUS, LAST_LOAD_TIME
  FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => '{{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL', START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())));


In [ ]:
-- The admin's "what needs my attention?" query
SELECT * FROM {{my_db}}.RAW.LOAD_MONITOR
WHERE ERROR_COUNT > 0 OR STATUS <> 'Loaded'
ORDER BY LAST_LOAD_TIME DESC;


> :mag_right: **Review the output.** The partial load surfaces here (ERROR_COUNT = 2). This one-line filter is the query you'd schedule or turn into an alert (next).

In [ ]:
-- Summarise all loads by status — for a quick visual
SELECT STATUS, COUNT(*) AS load_count
FROM {{my_db}}.RAW.LOAD_MONITOR
GROUP BY STATUS
ORDER BY load_count DESC;


> :bar_chart: **Make a chart (no code).** With the results above showing, click the **Chart** tab above the results grid. Set **Chart type = Bar**, **X-axis = STATUS**, **Y-axis = LOAD_COUNT**. You now have a load-health bar chart — `Loaded` vs `Partially loaded` at a glance.

🤖 **CoCo Prompt** — *how else could an admin monitor loads? click to copy:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Beyond INFORMATION_SCHEMA.COPY_HISTORY, what are my options for monitoring data loading health across a Snowflake account as an admin? Compare INFORMATION_SCHEMA.COPY_HISTORY vs SNOWFLAKE.ACCOUNT_USAGE.COPY_HISTORY (retention, latency, scope), and explain how ACCESS_HISTORY helps answer "who loaded this?".</pre>

---
### Part C · Grant the privileges that make loading (and unloading) work

Here's a support ticket you'll see constantly: *"I have access to the database but I can't load from the stage."* The cause is almost always that **stage and file-format privileges are separate from schema `USAGE`**:

| To do this... | ...the role needs |
|---|---|
| **Load** data (COPY INTO ← stage) | `READ` on the stage |
| **Unload** data (COPY INTO → stage, PUT) | `WRITE` on the stage |
| Use a named format in COPY | `USAGE` on the file format |
| Unload FROM (SELECT … FROM table) | `SELECT` on the source table (+ schema `USAGE`) |

Let's play it out. You'll create a stage you **own**, confirm your data-team role (`{{my_analyst_role}}`) is locked out, then grant exactly what it needs.

In [ ]:
-- A stage you own — for the team to both load from and unload to
CREATE STAGE IF NOT EXISTS {{my_db}}.RAW.TEAM_STAGE
  COMMENT = 'Shared load/unload stage for the data team';


In [ ]:
-- Who can use this stage right now?
SHOW GRANTS ON STAGE {{my_db}}.RAW.TEAM_STAGE;


> :mag_right: **Review the output.** Only the owning role appears. Your data-team role `{{my_analyst_role}}` is **not** listed — so right now, a team member using that role can't load from or unload to this stage, even though they can see the database. That's the blocked state.

In [ ]:
-- Grant load (READ) and unload (WRITE) on the stage to the data-team role
GRANT READ, WRITE ON STAGE {{my_db}}.RAW.TEAM_STAGE TO ROLE {{my_analyst_role}};

-- The unload cell below also needs to read from the source table — grant that too
GRANT USAGE  ON SCHEMA {{my_db}}.RAW                    TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE  {{my_db}}.RAW.SUPPLIER_PRODUCTS  TO ROLE {{my_analyst_role}};


Loading with a *named* file format also requires `USAGE` on that format — a commonly forgotten grant.

> :point_right: **Your turn.** Grant `USAGE` on your `CSV_SUPPLIER_FMT` file format to `{{my_analyst_role}}`, then run the SHOW GRANTS after it. Reference solution is just below the cell.

In [ ]:
-- Your turn: grant USAGE on CSV_SUPPLIER_FMT to {{my_analyst_role}}


> <details><summary>📖 Reference solution</summary>
>
>     GRANT USAGE ON FILE FORMAT {{my_db}}.RAW.CSV_SUPPLIER_FMT TO ROLE {{my_analyst_role}};
> </details>

In [ ]:
-- Confirm the data-team role now has stage access
SHOW GRANTS ON STAGE {{my_db}}.RAW.TEAM_STAGE;


> :mag_right: **Review the output.** `{{my_analyst_role}}` now appears with `READ` and `WRITE` — it can load from and unload to the stage. Blocked → granted → resolved.

> :bulb: **The gotcha to remember.** A role with schema `USAGE` can still hit *"Object does not exist or not authorized"* on a stage it can plainly see in `SHOW STAGES`. Schema `USAGE` = "you can see it exists"; stage `READ`/`WRITE` = "you can use it." Both are required — and loading with a named format needs format `USAGE` too.

#### Put the stage to work — prove the grant as the team 🔁

You just granted the team `READ` **and** `WRITE` on `TEAM_STAGE` — but a grant you haven't watched work is only a hopeful `GRANT` statement. Let's *prove* it: **switch into the data-team role** (`{{my_analyst_role}}`) — the very role that was locked out at the start of Part C — and do real work as that role.

**`WRITE` unlocks unloading — `COPY INTO` run in reverse.** Loading pulls files *into* a table; **unloading** (a.k.a. exporting or extracting) pushes query results *out* to files on a stage with `COPY INTO @stage FROM (SELECT …)`. The `SELECT` is the whole point — you're not dumping a raw table, you run *any* query and write exactly that shape to a file: pick the columns, cast the types, rename them on the way out. It's the standard way an admin hands another team or an external partner a clean, purpose-built extract **without** granting them access to the underlying tables. (`PUT` also writes to a stage, but from your local machine up — `COPY INTO @stage` is the in-database, query-driven version.)

`READ`, in turn, lets a role read those staged files back — and you don't even need a table for it; a `SELECT` can stream a file straight off the stage.

> :point_right: **The next cells run as `{{my_analyst_role}}`.** The role switch also picks up `PLATPLUS_WH` — the warehouse your team role is granted to use (your `{{my_wh}}` belongs to your admin role, not the team's). This is exactly the session a team member would have.

In [ ]:
-- Become the data-team role — the one that was blocked at the start of Part C.
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;

-- WRITE direction: unload query results to files on the stage.
-- FROM (SELECT ...) is the same transformation form COPY supports on load.
COPY INTO @{{my_db}}.RAW.TEAM_STAGE/exports/active_products
FROM (SELECT product_id, product_name, category, cost_per_unit
      FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS)
FILE_FORMAT = (TYPE = CSV FIELD_OPTIONALLY_ENCLOSED_BY = '"' COMPRESSION = NONE)
HEADER = TRUE
OVERWRITE = TRUE;


In [ ]:
-- Still the analyst: LIST reads the stage — the READ grant at work.
LIST @{{my_db}}.RAW.TEAM_STAGE;


> :mag_right: **Review the output.** Running as `{{my_analyst_role}}`, the unload reported `rows_unloaded` (50) and `LIST` shows a file like `exports/active_products_0_0_0.csv`. `HEADER = TRUE` wrote a header row; `FIELD_OPTIONALLY_ENCLOSED_BY = '"'` quotes any values containing commas; `COMPRESSION = NONE` keeps it plain `.csv` (Snowflake gzips by default). **The team role just wrote to the stage** — the `WRITE` grant in action, on the same role that couldn't touch this stage minutes ago.
>
> Next, still as the analyst, prove the `READ` direction: stream that file straight back with a `SELECT` — no table required.

In [ ]:
-- READ direction (still the analyst): point a SELECT straight at the file you just wrote.
-- Positional $1..$4 read the raw fields; the header comes back as the first row.
SELECT $1, $2, $3, $4
FROM @{{my_db}}.RAW.TEAM_STAGE/exports/
LIMIT 5;


> :point_right: **Switch back to your admin role.** You proved the grant as the team — now restore your own context so the Bonus section and the checks below run as your admin role again.

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};


> :mag_right: **Review the output.** The `SELECT` returned the header row (`"PRODUCT_ID"`, …) plus data rows — read **directly from the stage file**, no table needed. Every write and read you just ran executed as `{{my_analyst_role}}`: **WRITE** produced the file, **READ** streamed it back.

> :bulb: **This is why both grants exist — and why you tested as the team.** A role with only `READ` can consume extracts but never produce them; a role with `WRITE` can generate exports for partners and pipelines. Running the cells *as the analyst* is the real proof: the same role that hit *"not authorized"* at the top of Part C now moves files freely. Blocked → granted → **verified**.

---
### Bonus · Alert on failed loads

Monitoring you have to *remember to check* isn't really monitoring. The proactive step is to have Snowflake watch for you and act on its own — that's what an **alert** does.

A **[Snowflake alert](https://docs.snowflake.com/en/user-guide/alerts)** is a schema-level object built from three parts:
- **Condition** — a `SELECT` / `SHOW` / `CALL` that Snowflake evaluates on a schedule; if it returns **any rows**, the alert *fires*.
- **Action** — the SQL to run when it fires: send a notification, log a row, or call a stored procedure.
- **Schedule** — how often to check: every *n* minutes (`SCHEDULE = '60 MINUTE'`), or a `USING CRON` expression for calendar-based timing.

Two things to know before you build one:
- An alert is **created suspended** — it won't run until you `ALTER ALERT … RESUME` it (or trigger a one-off run with `EXECUTE ALERT`). That's a safety default so a half-finished alert never fires on its own.
- It needs **compute** — either a warehouse you name (what we'll do here) or Snowflake's *serverless* option (omit `WAREHOUSE` and Snowflake sizes the compute for you).

You'll build one that watches for problem loads, reusing the exact `COPY_HISTORY` filter from Part B. To keep the lab self-contained, its action writes a row to a **log table**; in production the action would send an email or Slack/Teams message via a **notification integration** (ask CoCo's `/notification` skill about that).

> :book: **Docs:** [Setting up alerts based on data in Snowflake](https://docs.snowflake.com/en/user-guide/alerts) · [`CREATE ALERT` reference](https://docs.snowflake.com/en/sql-reference/sql/create-alert)

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.LOAD_ALERT_LOG (
  alert_msg VARCHAR,
  logged_at TIMESTAMP_LTZ
);


In [ ]:
-- Condition: any committed load with errors on the partial table in the last 24h.
-- Action: log a row. Created SUSPENDED by default (it won't run on its own).
CREATE OR REPLACE ALERT {{my_db}}.RAW.ALERT_FAILED_LOADS
  WAREHOUSE = {{my_wh}}
  SCHEDULE  = '60 MINUTE'
  IF (EXISTS (
    SELECT 1
    FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
      TABLE_NAME => '{{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL',
      START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())))
    WHERE ERROR_COUNT > 0
  ))
  THEN
    INSERT INTO {{my_db}}.RAW.LOAD_ALERT_LOG (alert_msg, logged_at)
    VALUES ('Failed/partial loads detected in last 24h', CURRENT_TIMESTAMP());


The alert is created **suspended**, so it won't fire on its schedule. You don't have to wait an hour to test it: **`EXECUTE ALERT`** triggers a single run right now. (Triggering does not un-suspend it.)

In [ ]:
EXECUTE ALERT {{my_db}}.RAW.ALERT_FAILED_LOADS;


`EXECUTE ALERT` is **asynchronous** — it queues the run and returns immediately. Give it a few seconds, then run the cell below to see the logged row. If it's empty, run it again.

In [ ]:
SELECT * FROM {{my_db}}.RAW.LOAD_ALERT_LOG ORDER BY logged_at DESC;


> :mag_right: **Review the output.** A row appears: *"Failed/partial loads detected in last 24h."* The alert's condition matched your partial load and its action ran. You can also see the run's outcome (`STATE = TRIGGERED`) in `SNOWFLAKE.INFORMATION_SCHEMA.ALERT_HISTORY`.

> :bulb: **In production:** you'd resume the alert (`ALTER ALERT ... RESUME`) so it runs on its schedule, and its action would notify a person (email/Slack/PagerDuty) via a notification integration rather than log to a table. We leave it suspended here so it doesn't consume credits after the lab.

---
## Verify Your Work

In [ ]:
checks = [
    ('LOYALTY_SIGNUPS loaded',            f'SELECT COUNT(*) FROM {my_db}.RAW.LOYALTY_SIGNUPS', 1),
    ('SUPPLIER_PRODUCTS loaded',          f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS', 1),
    ('SUPPLIER_PRODUCTS_QUOTED fixed',    f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS_QUOTED', 1),
    ('SUPPLIER_PRODUCTS_PARTIAL loaded',  f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS_PARTIAL', 1),
]
all_passed = True
for label, sql, min_val in checks:
    try:
        result = session.sql(sql).collect()[0][0]
        ok = result >= min_val
        print(('✅' if ok else '⛔') + f' {label}: {result}')
        if not ok:
            all_passed = False
    except Exception as e:
        print(f'⛔ {label}: ERROR — {e}')
        all_passed = False

# Object checks (SHOW returns one row per matching object — real checks, not auto-pass)
for label, show_sql in [
    ('TEAM_STAGE exists',        f"SHOW STAGES LIKE 'TEAM_STAGE' IN SCHEMA {my_db}.RAW"),
    ('ALERT_FAILED_LOADS exists', f"SHOW ALERTS LIKE 'ALERT_FAILED_LOADS' IN SCHEMA {my_db}.RAW"),
]:
    try:
        n = len(session.sql(show_sql).collect())
        ok = n >= 1
        print(('✅' if ok else '⛔') + f' {label}: {n}')
        if not ok:
            all_passed = False
    except Exception as e:
        print(f'⛔ {label}: ERROR — {e}')
        all_passed = False

if all_passed:
    print('\n🎉 All checks passed.')


---
### Check your understanding 🧠

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q3")

---
### :snowflake: Key Takeaways

- If you don't define a file format, Snowflake assumes **CSV** (comma-delimited fields, newline-delimited rows) — spell out `TYPE` and `SKIP_HEADER` when your file differs
- A comma **inside a quoted field** splits the row and fails on column count — fix it with `FIELD_OPTIONALLY_ENCLOSED_BY = '"'`, never by silencing the error with `ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE` (that loads misaligned data)
- `COPY_HISTORY` records only **committed** loads — VALIDATION_MODE dry-runs and hard-aborts leave no trace; it has no "who" column (use `ACCESS_HISTORY` for that)
- The loads to watch for are the **quiet, partial** ones (`ON_ERROR=CONTINUE` with `ERROR_COUNT > 0`) — surface them with a monitoring view and an **alert**
- Loading needs stage `READ`; unloading needs stage `WRITE`; using a named format needs format `USAGE` — all **separate** from schema `USAGE`

**The Other Personas:**
- The **Data Analyst** notebook covers querying across LOYALTY_SIGNUPS and SUPPLIER_PRODUCTS
- The **Data Engineer** notebook covers ON_ERROR=CONTINUE, querying JSON with colon-dot path notation, and scheduled tasks

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.